# Phase 0: GPU check

**CRISP-DM stage:** Business understanding (project setup)

**Runs on:** Kaggle, with the GPU switched on. It uses no dataset and fits no model.

**Question it answers:** can a notebook from this project run on a Kaggle GPU with anomalib installed?

## 1. What Kaggle has before we install anything

anomalib 2.7.0 needs PyTorch 2.10 or newer, so we check Kaggle's version first. We read the version number without importing PyTorch, because importing it here could leave an old copy loaded in memory if the install below changes it.

In [ ]:
import platform
from importlib.metadata import PackageNotFoundError, version

print("Python before install:", platform.python_version())

# try/except so that a missing PyTorch is reported instead of stopping the notebook
try:
    print("PyTorch before install:", version("torch"))
except PackageNotFoundError:
    print("PyTorch before install: not installed")

## 2. Install anomalib

The exact version is pinned so that a later release cannot change results. No extra such as `[cu130]` is used, because anomalib's basic package does not list PyTorch, so Kaggle's own GPU build should stay in place.

In [ ]:
!pip install anomalib==2.7.0

## 3. Check the libraries and the GPU

`torch.cuda` is PyTorch's connection to NVIDIA GPUs. Seeing a GPU does not prove it works, so we also run one small calculation on it.

In [ ]:
import platform

import anomalib
import torch

# Collected in a list so the same lines can be printed and saved to a file
report_lines = []
report_lines.append("python: " + platform.python_version())
report_lines.append("anomalib: " + anomalib.__version__)
report_lines.append("pytorch: " + torch.__version__)

gpu_available = torch.cuda.is_available()
report_lines.append("gpu available: " + str(gpu_available))

if gpu_available:
    report_lines.append("gpu count: " + str(torch.cuda.device_count()))
    # 0 means the first GPU
    report_lines.append("gpu name: " + torch.cuda.get_device_name(0))

    # 1000 by 1000 is big enough to be real GPU work and should still finish quickly
    matrix = torch.rand(1000, 1000, device="cuda")
    product = matrix @ matrix
    report_lines.append("calculation ran on: " + str(product.device))
else:
    # Reported rather than raised, so the report is still saved below
    report_lines.append("calculation ran on: skipped, no GPU")

for line in report_lines:
    print(line)

## 4. Save the results to the notebook's output

Files saved in the working folder become the notebook's output, so they survive a disconnect. The package list records exactly what Kaggle used, which can differ from the laptop.

In [ ]:
import subprocess
import sys

with open("gpu_check_report.txt", "w") as report_file:
    for line in report_lines:
        report_file.write(line + "\n")

# sys.executable is the Python running this notebook, so pip lists its packages
freeze_result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
)

with open("kaggle_pip_freeze.txt", "w") as freeze_file:
    freeze_file.write(freeze_result.stdout)

print("Saved gpu_check_report.txt and kaggle_pip_freeze.txt")